In [1]:
%pip install geopy tqdm


   ---------------------------------------- 0/2 [geographiclib]
   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

# 1) 샘플 파일 로드
tow = pd.read_csv("tow_addresses_cleaned_sample300.csv", encoding="utf-8-sig")
park = pd.read_csv("parking_addresses_cleaned_sample200.csv", encoding="utf-8-sig")

# 2) 지오코더 (무료 / 속도제한 준수 필수)
geolocator = Nominatim(user_agent="pm_seoul_geocoding")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2, return_value_on_exception=None)

def run_geocode(df, addr_col):
    lats, lons, ok = [], [], []
    for a in tqdm(df[addr_col].fillna("").tolist()):
        if not a.strip():
            lats.append(None); lons.append(None); ok.append(False); continue
        loc = geocode(a)
        if loc:
            lats.append(loc.latitude); lons.append(loc.longitude); ok.append(True)
        else:
            lats.append(None); lons.append(None); ok.append(False)
    out = df.copy()
    out["lat"] = lats
    out["lon"] = lons
    out["geocode_ok"] = ok
    return out

# 3) 견인: addr_clean 기준
tow_g = run_geocode(tow, "addr_clean")

# 4) 주차구역: addr_plus(주소+상세위치) 우선, 실패하면 addr_clean 재시도(옵션)
park_g = run_geocode(park, "addr_plus")
fail = park_g["geocode_ok"] == False
if fail.any():
    retry = run_geocode(park_g.loc[fail].drop(columns=["lat","lon","geocode_ok"]), "addr_clean")
    park_g.loc[fail, ["lat","lon","geocode_ok"]] = retry[["lat","lon","geocode_ok"]].values

# 5) 저장
tow_g.to_csv("tow_geocoded_sample300.csv", index=False, encoding="utf-8-sig")
park_g.to_csv("parking_geocoded_sample200.csv", index=False, encoding="utf-8-sig")

# 6) 성공률 출력
print("견인 성공률:", tow_g["geocode_ok"].mean())
print("주차구역 성공률:", park_g["geocode_ok"].mean())


100%|████████████████████████████████████████████████████████████████████████████████| 198/198 [03:57<00:00,  1.20s/it]

견인 성공률: 0.96
주차구역 성공률: 0.82


In [3]:
%pip install folium


Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import folium

tow = pd.read_csv("tow_geocoded_sample300.csv", encoding="utf-8-sig")
park = pd.read_csv("parking_geocoded_sample200.csv", encoding="utf-8-sig")

tow_ok = tow[tow["geocode_ok"] == True].dropna(subset=["lat","lon"]).copy()
park_ok = park[park["geocode_ok"] == True].dropna(subset=["lat","lon"]).copy()

# BallTree는 라디안 단위
tow_rad = np.deg2rad(tow_ok[["lat","lon"]].values)
park_rad = np.deg2rad(park_ok[["lat","lon"]].values)

tree = BallTree(park_rad, metric="haversine")
# 100m 반경: (100 / 지구반지름 6371000)
radius = 100 / 6371000
dist, ind = tree.query(tow_rad, k=1)
tow_ok["near_parking_100m"] = (dist[:,0] <= radius)

m = folium.Map(location=[37.5665, 126.9780], zoom_start=11, tiles="cartodbpositron")

# 주차구역(파란 점)
for _, r in park_ok.iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=4, tooltip="주차구역", fill=True).add_to(m)

# 견인(주차구역 근처=빨강, 아니면 회색)
for _, r in tow_ok.iterrows():
    color = "red" if r["near_parking_100m"] else "gray"
    folium.CircleMarker([r["lat"], r["lon"]], radius=4, color=color, fill=True, fill_opacity=0.7).add_to(m)

m.save("seoul_tow_near_parking_100m_sample.html")
print("완료: seoul_tow_near_parking_100m_sample.html")


완료: seoul_tow_near_parking_100m_sample.html


In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# 1) 지오코딩 결과 로드
tow = pd.read_csv("tow_geocoded_sample300.csv", encoding="utf-8-sig")
park = pd.read_csv("parking_geocoded_sample200.csv", encoding="utf-8-sig")

# 2) 성공한 좌표만 필터
tow_ok = tow[tow["geocode_ok"] == True].dropna(subset=["lat", "lon"]).copy()
park_ok = park[park["geocode_ok"] == True].dropna(subset=["lat", "lon"]).copy()

# 3) 지도 기본 중심(서울시청 근처)
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11, tiles="cartodbpositron")

# 4) 견인 점(클러스터)
tow_cluster = MarkerCluster(name="견인(샘플)").add_to(m)
for _, r in tow_ok.iterrows():
    tooltip = f"견인 | {r.get('구정보','')} | {r.get('유형','')} | {r.get('주소','')}"
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=4,
        tooltip=tooltip,
        fill=True,
        fill_opacity=0.7,
    ).add_to(tow_cluster)

# 5) 주차구역 점(클러스터)
park_cluster = MarkerCluster(name="주차구역(샘플)").add_to(m)
for _, r in park_ok.iterrows():
    tooltip = f"주차구역 | {r.get('시군구명','')} | {r.get('거치대 유무','')} | {r.get('주소','')}"
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=5,
        tooltip=tooltip,
        fill=True,
        fill_opacity=0.7,
    ).add_to(park_cluster)

# 6) 레이어 토글
folium.LayerControl(collapsed=False).add_to(m)

# 7) 저장
m.save("seoul_pm_tow_vs_parking_sample_map.html")
print("완료: seoul_pm_tow_vs_parking_sample_map.html")


완료: seoul_pm_tow_vs_parking_sample_map.html


In [6]:
print("완료: seoul_pm_tow_vs_parking_sample_map.html")

완료: seoul_pm_tow_vs_parking_sample_map.html


In [7]:
from IPython.display import IFrame, display

display(IFrame("seoul_pm_tow_vs_parking_sample_map.html", width=1000, height=700))


In [1]:
import os
import time
import json
import pandas as pd
import requests
from tqdm import tqdm
import folium
from folium.plugins import MarkerCluster

CENTERS_CSV = "서울시_자치구_중심점_2017.csv"
PARKING_CSV = "서울시 전동킥보드 주차구역 현황.csv"

OUT_CSV = "parking_with_latlon.csv"
OUT_MAP = "seoul_kickboard_parking_map.html"

CACHE_PATH = "geocode_cache.json"

USER_AGENT = "seoul-kickboard-parking-mapper/1.0 (contact: your_email@example.com)"
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

def load_cache(path=CACHE_PATH):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(cache, path=CACHE_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

def geocode_address(address, cache, session, sleep_sec=1.1, max_retries=3):
    """
    Nominatim 지오코딩: 주소 -> (lat, lon)
    - 캐시 사용
    - 재시도 + 속도 제한
    """
    address = str(address).strip()
    if not address:
        return None, None

    if address in cache:
        lat, lon = cache[address]
        return lat, lon

    params = {
        "q": address,
        "format": "json",
        "limit": 1
    }

    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(NOMINATIM_URL, params=params, timeout=15)
            if r.status_code == 429:
                # Too Many Requests
                time.sleep(5 * attempt)
                continue
            r.raise_for_status()
            data = r.json()

            if data:
                lat = float(data[0]["lat"])
                lon = float(data[0]["lon"])
                cache[address] = [lat, lon]
                time.sleep(sleep_sec)
                return lat, lon
            else:
                cache[address] = [None, None]
                time.sleep(sleep_sec)
                return None, None

        except Exception:
            time.sleep(2 * attempt)

    cache[address] = [None, None]
    return None, None

def main():
    # 1) 데이터 로드 (cp949)
    centers = pd.read_csv(CENTERS_CSV, encoding="cp949")
    parking = pd.read_csv(PARKING_CSV, encoding="cp949")

    # centers: X=경도, Y=위도
    centers = centers.rename(columns={"X": "center_lon", "Y": "center_lat"})

    # 2) 주차구역 주소 만들기 (서울특별시 + 구 + 주소 + 상세위치)
    # (지오코딩 성공률을 올리기 위해 최대한 자세히)
    parking["full_address"] = (
        "서울특별시 " + parking["시군구명"].astype(str).str.strip() + " " +
        parking["주소"].astype(str).str.strip() + " " +
        parking["상세위치"].astype(str).str.strip()
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    # 3) 지오코딩
    cache = load_cache()
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    lats = []
    lons = []

    print("지오코딩 진행 중... (처음 한 번은 시간이 걸립니다. 캐시가 쌓이면 빨라집니다.)")
    for addr in tqdm(parking["full_address"].tolist()):
        lat, lon = geocode_address(addr, cache, session)
        lats.append(lat)
        lons.append(lon)

    parking["lat"] = lats
    parking["lon"] = lons

    save_cache(cache)

    # 4) 좌표 성공/실패 통계
    success = parking["lat"].notna().sum()
    total = len(parking)
    print(f"지오코딩 성공: {success}/{total} ({success/total:.1%})")

    # 5) 자치구 중심점 병합 (시각화 참고용)
    merged = parking.merge(
        centers[["시군구명", "center_lat", "center_lon"]],
        on="시군구명",
        how="left"
    )

    # 6) 결과 CSV 저장
    merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"좌표 포함 CSV 저장: {OUT_CSV}")

    # 7) 지도 생성 (folium)
    # 지도 중심은 서울 대략 중심(또는 중심점 평균)
    map_center_lat = centers["center_lat"].mean()
    map_center_lon = centers["center_lon"].mean()

    m = folium.Map(location=[map_center_lat, map_center_lon], zoom_start=11)

    # 자치구 중심점 표시(파란 원)
    for _, row in centers.iterrows():
        folium.CircleMarker(
            location=[row["center_lat"], row["center_lon"]],
            radius=5,
            popup=f'자치구 중심: {row["시군구명"]}',
            fill=True
        ).add_to(m)

    # 주차구역 점 표시(클러스터)
    cluster = MarkerCluster(name="전동킥보드 주차구역").add_to(m)

    # 좌표 있는 것만 찍기
    has_coord = merged.dropna(subset=["lat", "lon"]).copy()

    for _, row in has_coord.iterrows():
        popup = (
            f"<b>{row['시군구명']}</b><br>"
            f"{row['주소']}<br>"
            f"{row['상세위치']}<br>"
            f"거치대 유무: {row['거치대 유무']}"
        )
        folium.Marker(
            location=[row["lat"], row["lon"]],
            popup=folium.Popup(popup, max_width=300)
        ).add_to(cluster)

    folium.LayerControl().add_to(m)

    m.save(OUT_MAP)
    print(f"지도 HTML 저장: {OUT_MAP}")
    print("끝! 이제 HTML을 더블클릭해서 열면 전체 주차구역이 지도에 표시됩니다.")

if __name__ == "__main__":
    main()


지오코딩 진행 중... (처음 한 번은 시간이 걸립니다. 캐시가 쌓이면 빨라집니다.)


100%|██████████████████████████████████████████████████████████████████████████████| 329/329 [1:37:41<00:00, 17.82s/it]

지오코딩 성공: 0/329 (0.0%)
좌표 포함 CSV 저장: parking_with_latlon.csv
지도 HTML 저장: seoul_kickboard_parking_map.html
끝! 이제 HTML을 더블클릭해서 열면 전체 주차구역이 지도에 표시됩니다.
